In [2]:
import torch

In [3]:
data = torch.load('data_tensor_2.pt')

In [47]:
data.shape

torch.Size([1749370, 2, 23])

In [5]:
def return_accuracy(w):
    if type(w) is not torch.Tensor:
        w = torch.tensor(w, dtype=torch.float32)
    y_true = torch.zeros(data.shape[0], dtype=torch.long)
    U = torch.matmul(data, w)                
    y_pred = torch.argmax(U, dim=1)      
    correct = (U[:, 0] - U[:, 1]) > 0
    accuracy = correct.float().mean()
    return accuracy.item()

In [6]:
import numpy as np

def interpret_vector(weight_vector, normalize=True):
    """
    Interprets a weight vector for the trolley problem utility function.
    
    Args:
        weight_vector (list/array): Weights corresponding to features
        normalize (bool): Whether to normalize people weights to show relative preferences
        
    Returns:
        str: Human-readable interpretation of the moral preferences encoded in the weights
    """
    
    # Feature names for reference
    feature_names = ['Intervention', 'Barrier', 'CrossingSignal', 'Man', 'Woman', 
                    'Pregnant', 'Stroller', 'OldMan', 'OldWoman', 'Boy', 'Girl', 'Homeless', 
                    'LargeWoman', 'LargeMan', 'Criminal', 'MaleExecutive', 'FemaleExecutive', 
                    'FemaleAthlete', 'MaleAthlete', 'FemaleDoctor', 'MaleDoctor', 'Dog', 'Cat']
    
    if len(weight_vector) != len(feature_names):
        raise ValueError(f"Weight vector length ({len(weight_vector)}) doesn't match feature count ({len(feature_names)})")
    
    weights = np.array(weight_vector)
    
    # Separate structural and people weights
    structural_weights = weights[:3]
    people_weights = weights[3:]
    people_names = feature_names[3:]
    
    # Normalize people weights if requested
    if normalize and len(people_weights) > 0:
        people_sum = np.sum(np.abs(people_weights))
        if people_sum > 0:
            normalized_people = people_weights / people_sum
        else:
            normalized_people = people_weights
    else:
        normalized_people = people_weights
    
    output_lines = []
    output_lines.append("=== TROLLEY PROBLEM MORAL PREFERENCES ===\n")
    
    # Analyze structural preferences
    interv_weight = structural_weights[0]
    barrier_weight = structural_weights[1]
    crossing_weight = structural_weights[2]
    
    structural_insights = []
    
    if abs(interv_weight) > 0.05:
        if interv_weight > 0:
            structural_insights.append(f"Favors taking action over inaction (weight: {interv_weight:.2f})")
        else:
            structural_insights.append(f"Prefers inaction over intervention (weight: {interv_weight:.2f})")
    
    if abs(barrier_weight) > 0.05:
        if barrier_weight > 0:
            structural_insights.append(f"Values passengers over pedestrians (weight: {barrier_weight:.2f})")
        else:
            structural_insights.append(f"Values pedestrians over passengers (weight: {barrier_weight:.2f})")
    
    if abs(crossing_weight) > 0.05:
        if crossing_weight > 0:
            structural_insights.append(f"Strongly considers legal vs illegal crossing (weight: {crossing_weight:.2f})")
        else:
            structural_insights.append(f"Penalizes legal behavior - counterintuitive (weight: {crossing_weight:.2f})")
    
    if structural_insights:
        output_lines.append("STRUCTURAL PREFERENCES:")
        for insight in structural_insights:
            output_lines.append(f"  • {insight}")
        output_lines.append("")
    
    # Analyze people preferences
    if len(normalized_people) > 0:
        # Sort by preference strength
        sorted_indices = np.argsort(normalized_people)[::-1]  # Descending order
        
        # Group into categories
        highly_valued = []
        moderately_valued = []
        devalued = []
        neutral = []
        
        for idx in sorted_indices:
            name = people_names[idx]
            weight = normalized_people[idx]
            
            if weight > 0.08:  # Highly valued
                highly_valued.append((name, weight))
            elif weight > 0.05:  # Moderately valued
                moderately_valued.append((name, weight))
            else:  # Devalued
                devalued.append((name, weight))
        
        output_lines.append("PERSON TYPE PREFERENCES:")
        
        if highly_valued:
            output_lines.append("  HIGHLY VALUED:")
            for name, weight in highly_valued:
                output_lines.append(f"    • {name}: {weight:.3f}")
        
        if moderately_valued:
            output_lines.append("  MODERATELY VALUED:")
            for name, weight in moderately_valued:
                output_lines.append(f"    • {name}: {weight:.3f}")
        
        if devalued:
            output_lines.append("  DEVALUED:")
            for name, weight in devalued:
                output_lines.append(f"    • {name}: {weight:.3f}")
        
        output_lines.append("")
    
    # Generate key insights
    insights = []
    
    # Check for demographic patterns
    children_weights = [normalized_people[people_names.index(name)] for name in ['Boy', 'Girl', 'Stroller'] if name in people_names]
    adult_weights = [normalized_people[people_names.index(name)] for name in ['Man', 'Woman'] if name in people_names]
    elderly_weights = [normalized_people[people_names.index(name)] for name in ['OldMan', 'OldWoman'] if name in people_names]
    
    if children_weights and adult_weights:
        avg_children = np.mean(children_weights)
        avg_adults = np.mean(adult_weights)
        if avg_children > avg_adults + 0.02:
            insights.append(f"Shows strong preference for children over adults (children: {avg_children:.3f}, adults: {avg_adults:.3f})")
        elif avg_adults > avg_children + 0.02:
            insights.append(f"Prioritizes adults over children (adults: {avg_adults:.3f}, children: {avg_children:.3f})")
    
    # Check professional bias
    professionals = ['FemaleDoctor', 'MaleDoctor', 'FemaleExecutive', 'MaleExecutive']
    prof_weights = [normalized_people[people_names.index(name)] for name in professionals if name in people_names]
    if prof_weights:
        avg_prof = np.mean(prof_weights)
        if avg_prof > 0.06:
            insights.append(f"Values professionals highly (avg: {avg_prof:.3f})")
    
    # Check for gender bias
    male_types = ['Man', 'OldMan', 'Boy', 'LargeMan', 'MaleExecutive', 'MaleAthlete', 'MaleDoctor']
    female_types = ['Woman', 'OldWoman', 'Girl', 'LargeWoman', 'FemaleExecutive', 'FemaleAthlete', 'FemaleDoctor']
    
    male_weights = [normalized_people[people_names.index(name)] for name in male_types if name in people_names]
    female_weights = [normalized_people[people_names.index(name)] for name in female_types if name in people_names]
    
    if male_weights and female_weights:
        avg_male = np.mean(male_weights)
        avg_female = np.mean(female_weights)
        if abs(avg_male - avg_female) > 0.02:
            if avg_male > avg_female:
                insights.append(f"Shows slight male bias (male: {avg_male:.3f}, female: {avg_female:.3f})")
            else:
                insights.append(f"Shows slight female bias (female: {avg_female:.3f}, male: {avg_male:.3f})")
    
    # Check for vulnerable populations
    vulnerable = ['Homeless', 'Criminal']
    vuln_weights = [normalized_people[people_names.index(name)] for name in vulnerable if name in people_names]
    if vuln_weights:
        avg_vuln = np.mean(vuln_weights)
        if avg_vuln < -0.01:
            insights.append(f"Devalues marginalized groups (avg: {avg_vuln:.3f})")
    
    # Check animal consideration
    animals = ['Dog', 'Cat']
    animal_weights = [normalized_people[people_names.index(name)] for name in animals if name in people_names]
    if animal_weights:
        avg_animal = np.mean(animal_weights)
        if avg_animal > 0.02:
            insights.append(f"Assigns significant value to animal lives (avg: {avg_animal:.3f})")
    
    if insights:
        output_lines.append("KEY MORAL INSIGHTS:")
        for insight in insights:
            output_lines.append(f"  • {insight}")
        output_lines.append("")
    
    # Overall moral philosophy summary
    output_lines.append("MORAL PHILOSOPHY SUMMARY:")
    
    # Determine overall approach
    if interv_weight > 0.1:
        output_lines.append("  • Utilitarian approach - willing to actively intervene to maximize outcomes")
    elif interv_weight < -0.1:
        output_lines.append("  • Deontological approach - prefers not to actively cause harm")
    else:
        output_lines.append("  • Mixed approach to intervention vs. inaction")
    
    if crossing_weight > 0.1:
        output_lines.append("  • Rule-based ethics - values legal and social compliance")
    
    # Find the most extreme preference
    if len(normalized_people) > 0:
        max_idx = np.argmax(np.abs(normalized_people))
        max_weight = normalized_people[max_idx]
        max_name = people_names[max_idx]
        
        if max_weight > 0:
            output_lines.append(f"  • Strongest positive bias: {max_name} (weight: {max_weight:.3f})")
        else:
            output_lines.append(f"  • Strongest negative bias: {max_name} (weight: {max_weight:.3f})")
    
    output_lines.append(f"\nUtility Calculation: Higher scores = more likely to save that option")
    output_lines.append("Decision Rule: Choose the option (save vs. alternative) with higher utility score")
    
    return "\n".join(output_lines)


In [179]:
# Openai

vector = [-0.3, -0.5, 0.4, 1.0, 1.0, 1.6, 1.8, 0.7, 0.7, 1.5, 1.5, 0.5, 0.8, 0.8, -0.6, 1.2, 1.2, 1.4, 1.4, 1.3, 1.3, 0.6, 0.6]
print(return_accuracy(vector))
print(interpret_vector(vector))

0.7666988968849182
=== TROLLEY PROBLEM MORAL PREFERENCES ===

STRUCTURAL PREFERENCES:
  • Prefers inaction over intervention (weight: -0.30)
  • Values pedestrians over passengers (weight: -0.50)
  • Strongly considers legal vs illegal crossing (weight: 0.40)

PERSON TYPE PREFERENCES:
  HIGHLY VALUED:
    • Stroller: 0.084
  MODERATELY VALUED:
    • Pregnant: 0.074
    • Girl: 0.070
    • Boy: 0.070
    • MaleAthlete: 0.065
    • FemaleAthlete: 0.065
    • MaleDoctor: 0.060
    • FemaleDoctor: 0.060
    • FemaleExecutive: 0.056
    • MaleExecutive: 0.056
    • Man: 0.047
    • Woman: 0.047
    • LargeWoman: 0.037
    • LargeMan: 0.037
    • OldWoman: 0.033
    • OldMan: 0.033

KEY MORAL INSIGHTS:
  • Shows strong preference for children over adults (children: 0.074, adults: 0.047)
  • Assigns significant value to animal lives (avg: 0.028)

MORAL PHILOSOPHY SUMMARY:
  • Deontological approach - prefers not to actively cause harm
  • Rule-based ethics - values legal and social compliance

# Method 1 : Random Sampling

In [36]:
import numpy as np
import time

def random_search_optimizer(n_dimensions=23, n_iterations=10000, bounds=(-10.0, 10.0)):
    """
    Performs a random search to find the best weight vector.

    Args:
        n_dimensions (int): The number of dimensions in the weight vector.
        n_iterations (int): The number of random vectors to test.
        bounds (tuple): A tuple (min_val, max_val) for the random weights.

    Returns:
        tuple: A tuple containing (best_weights, best_accuracy).
    """
    print("--- Starting Random Search ---")
    start_time = time.time()
    
    best_accuracy = -1.0
    best_weights = None

    for i in range(n_iterations):
        # Generate a random weight vector within the specified bounds
        weights = np.random.uniform(bounds[0], bounds[1], n_dimensions)
        
        # Get the accuracy for this vector
        accuracy = return_accuracy(weights)
        
        # If it's the best we've seen, save it
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_weights = weights
            print(f"Iteration {i+1}/{n_iterations}: New best accuracy = {accuracy:.6f}")

    end_time = time.time()
    print(f"--- Random Search Finished in {end_time - start_time:.2f} seconds ---")
    return best_weights, best_accuracy

# Method 2 - Hill Climbing

In [77]:
import numpy as np
import time

def hill_climbing_optimizer(n_dimensions=23, bounds=(-10.0, 10.0), step_size=0.01, initial_weights=None, patience=1000, min_improvement=0.000001):
    """
    Performs a hill climbing search with early stopping.

    Args:
        n_dimensions (int): The number of dimensions.
        bounds (tuple): A tuple (min_val, max_val) for the weights.
        step_size (float): The magnitude of the random change at each step.
        initial_weights (array): Starting weights, if None uses random initialization.
        patience (int): Number of iterations without improvement before stopping.
        min_improvement (float): Minimum improvement required to reset patience counter.

    Returns:
        tuple: A tuple containing (best_weights, best_accuracy).
    """
    print("--- Starting Hill Climbing ---")
    start_time = time.time()
    
    # 1. Start with a random vector or provided initial weights
    if initial_weights is not None:
        current_weights = np.array(initial_weights)
    else:
        current_weights = np.random.uniform(bounds[0], bounds[1], n_dimensions)
    current_accuracy = return_accuracy(current_weights)
    print(f"Initial accuracy: {current_accuracy:.6f}")

    # 2. Initialize patience counter
    iterations_without_improvement = 0
    iteration = 0

    # 3. Loop until patience is exhausted
    while iterations_without_improvement < patience:
        # Create a new vector by making a small, random change (a "step")
        noise = np.random.normal(0, step_size, n_dimensions)
        new_weights = current_weights + noise
        
        # Clip the values to stay within the defined bounds
        new_weights = np.clip(new_weights, bounds[0], bounds[1])

        # Evaluate the new vector
        new_accuracy = return_accuracy(new_weights)
        
        # If the new vector is better by at least min_improvement, move to that position
        if new_accuracy > current_accuracy + min_improvement:
            current_weights = new_weights
            current_accuracy = new_accuracy
            iterations_without_improvement = 0  # Reset patience counter
            print(f"Iteration {iteration+1}: New best accuracy = {current_accuracy:.6f}")
        else:
            iterations_without_improvement += 1
        
        iteration += 1

    end_time = time.time()
    print(f"--- Hill Climbing Finished in {end_time - start_time:.2f} seconds after {iteration} iterations ---")
    print(f"Stopped due to {patience} iterations without improvement >= {min_improvement}")
    return current_weights, current_accuracy

# Method 3: Genetic Algorithm

In [ ]:
import random
import time
import numpy as np
from deap import base, creator, tools, algorithms

# (Mock return_accuracy function would be here)

# --- DEAP Setup ---
# We are trying to MAXIMIZE accuracy, so weights are 1.0
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()

# Attribute generator: each weight is a float between -10 and 10
N_DIMENSIONS = 23
BOUND_LOW, BOUND_UP = -10.0, 10.0
toolbox.register("attr_float", random.uniform, BOUND_LOW, BOUND_UP)

# Structure initializers
# An "Individual" is a list of N_DIMENSIONS floats
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_float, n=N_DIMENSIONS)
# A "population" is a list of individuals
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

# --- Genetic Operators ---
# Evaluation function
def eval_accuracy(individual):
    # DEAP works with lists, but our function expects a numpy array
    weights = np.array(individual)
    return (return_accuracy(weights),) # Must return a tuple

toolbox.register("evaluate", eval_accuracy)
# Crossover operator
toolbox.register("mate", tools.cxTwoPoint)
# Mutation operator
toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=0.1, indpb=0.1)
# Selection operator
toolbox.register("select", tools.selTournament, tournsize=3)

def genetic_algorithm_optimizer(pop_size=100, n_generations=200):
    """
    Uses a Genetic Algorithm to find the best weight vector.
    """
    print("--- Starting Genetic Algorithm ---")
    start_time = time.time()

    pop = toolbox.population(n=pop_size)
    # Track the best individual found
    hof = tools.HallOfFame(1)
    # Gather statistics
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("std", np.std)
    stats.register("min", np.min)
    stats.register("max", np.max)

    # Run the evolutionary algorithm
    algorithms.eaSimple(pop, toolbox, cxpb=0.5, mutpb=0.2, ngen=n_generations, 
                        stats=stats, halloffame=hof, verbose=True)

    end_time = time.time()
    print(f"--- Genetic Algorithm Finished in {end_time - start_time:.2f} seconds ---")
    
    best_individual = hof[0]
    best_accuracy = best_individual.fitness.values[0]
    best_weights = np.array(best_individual)
    
    return best_weights, best_accuracy

# Method 4 - Bayesian Optimization

In [39]:
import numpy as np
import time
from skopt import gp_minimize
from skopt.space import Real


def bayesian_optimizer(n_dimensions=23, n_calls=200, bounds=(-10.0, 10.0)):
    """
    Uses Bayesian Optimization to find the best weight vector.

    Args:
        n_dimensions (int): The number of dimensions.
        n_calls (int): The number of times to call return_accuracy.
        bounds (tuple): A tuple (min_val, max_val) for the weights.

    Returns:
        tuple: A tuple containing (best_weights, best_accuracy).
    """
    print("--- Starting Bayesian Optimization ---")
    start_time = time.time()
    
    # 1. Define the search space
    search_space = [Real(bounds[0], bounds[1], name=f'w_{i}') for i in range(n_dimensions)]

    # 2. Define the objective function
    # scikit-optimize performs MINIMIZATION, so we must return 1.0 - accuracy.
    def objective_function(weights):
        weights = np.array(weights)
        accuracy = return_accuracy(weights)
        return 1.0 - accuracy

    # 3. Run the optimizer
    result = gp_minimize(
        func=objective_function,
        dimensions=search_space,
        n_calls=n_calls,
        n_initial_points=50, # How many random points to probe before building the model
        random_state=np.random.randint(0, 10000),
        verbose=True
    )

    end_time = time.time()
    print(f"--- Bayesian Optimization Finished in {end_time - start_time:.2f} seconds ---")
    
    # The result.x contains the best parameters found
    best_weights = np.array(result.x)
    # The result.fun contains the best *minimized* value (1.0 - accuracy)
    best_accuracy = 1.0 - result.fun
    
    return best_weights, best_accuracy


# Method 5 - CMA-ES

In [57]:
import numpy as np
import cma
import time


def cma_es_optimizer(n_dimensions=23, initial_guess=None, sigma=6, n_iterations=1000, bounds = [-100, 100]):
    """
    Uses the CMA-ES algorithm to find the best weight vector.

    Args:
        n_dimensions (int): The number of dimensions.
        initial_guess (np.ndarray): An initial starting point for the search. If None, starts at all zeros.
        sigma (float): The initial standard deviation (step size). This is a crucial parameter.
        n_iterations (int): The maximum number of iterations (function evaluations).

    Returns:
        tuple: A tuple containing (best_weights, best_accuracy).
    """
    print("--- Starting CMA-ES ---")
    start_time = time.time()

    if initial_guess is None:
        initial_guess = np.zeros(n_dimensions)

    # CMA-ES aims to MINIMIZE a function. Because our function returns
    # accuracy (higher is better), we need to minimize its negative.
    # We create a wrapper "loss" function for this purpose.
    def loss_function(weights):
        accuracy = return_accuracy(weights)
        return -accuracy # Return the negative, so minimizing it maximizes accuracy

    # Create a CMA-ES optimizer instance
    # The 'inopts' dictionary allows you to set options like bounds.
    es = cma.CMAEvolutionStrategy(initial_guess, sigma, {'bounds': bounds})

    # The optimization loop
    # We can use the 'ask-and-tell' interface for more control.
    iterations = 0
    while not es.stop() and iterations < n_iterations:
        # Ask for a new population of solutions
        solutions = es.ask()
        
        # Tell the optimizer the fitness (loss) of each solution
        # This requires calling your function for each candidate vector
        fitness_values = [loss_function(w) for w in solutions]
        es.tell(solutions, fitness_values)
        
        # Log progress and increment counter
        es.disp()
        iterations += 1

    end_time = time.time()
    print(f"--- CMA-ES Finished in {end_time - start_time:.2f} seconds ---")

    # The result is stored in the `result` property of the optimizer
    best_weights = es.result.xbest
    # The best fitness is the minimized loss (a negative number)
    best_loss = es.result.fbest
    # Convert it back to accuracy
    best_accuracy = -best_loss

    return best_weights, best_accuracy


# Jury experiment

In [74]:
def create_jury(n_per_method=5):
    """
    Creates a jury of weight vectors by running different optimization methods.
    Returns a dictionary with algorithm names as keys and lists of weight vectors as values.
    """
    jury = {
        'random_hill': [],
        'genetic': [],
        'bayesian': [],
        'cma_es': []
    }
    
    print("--- Assembling the Jury ---")
    
    # 1. Random Search + Hill Climbing
    print("\n--- Generating 5 vectors from Random Search + Hill Climbing ---")
    for i in range(n_per_method):
        print(f"\nJury Member {i+1}/{n_per_method} (Random+Hill)")
        random_weights, _ = random_search_optimizer(n_iterations=10000)
        hill_weights, _ = hill_climbing_optimizer(initial_weights=random_weights)
        jury['random_hill'].append(hill_weights)

    # 2. Genetic Algorithm
    print("\n--- Generating 5 vectors from Genetic Algorithm ---")
    for i in range(n_per_method):
        print(f"\nJury Member {i+1}/{n_per_method} (GA)")
        genetic_weights, _ = genetic_algorithm_optimizer(pop_size=200, n_generations=300)
        jury['genetic'].append(genetic_weights)

    # 3. Bayesian Optimization
    print("\n--- Generating 5 vectors from Bayesian Optimization ---")
    for i in range(n_per_method):
        print(f"\nJury Member {i+1}/{n_per_method} (Bayesian)")
        bayes_weights, _ = bayesian_optimizer(n_calls=100)
        jury['bayesian'].append(bayes_weights)

    # 4. CMA-ES
    print("\n--- Generating 5 vectors from CMA-ES ---")
    for i in range(n_per_method):
        print(f"\nJury Member {i+1}/{n_per_method} (CMA-ES)")
        cma_weights, _ = cma_es_optimizer(n_iterations=10000, sigma=6, bounds = [-10, 10])
        jury['cma_es'].append(cma_weights)
        
    print("\n--- Jury Assembly Complete ---")
    print(f"Total jury members: {sum(len(members) for members in jury.values())}")
    for method, members in jury.items():
        print(f"  {method}: {len(members)} members")
    
    return jury


In [75]:
def calculate_jury_accuracy(jury, data_tensor):
    """
    Calculates the accuracy of a jury of weight vectors using majority voting.
    """
    n_samples = data_tensor.shape[0]
    all_votes = []

    # Get votes from each jury member
    for i, member_weights in enumerate(jury):
        print(f"Collecting votes from Jury Member {i+1}/{len(jury)}...")
        if type(member_weights) is not torch.Tensor:
            member_weights = torch.tensor(member_weights, dtype=torch.float32)
        
        U = torch.matmul(data_tensor, member_weights)
        # A vote for 0 means U[:, 0] > U[:, 1], a vote for 1 means the opposite
        votes = (U[:, 1] > U[:, 0]).long()
        all_votes.append(votes)
    
    # Stack votes for easy majority calculation
    votes_tensor = torch.stack(all_votes)
    
    # Calculate majority vote for each sample
    # A majority vote of 0 means most jury members preferred the first option.
    # A majority vote of 1 means most jury members preferred the second option.
    majority_votes, _ = torch.mode(votes_tensor, dim=0)
    
    # Ground truth is always 0 (the first option is better)
    y_true = torch.zeros(n_samples, dtype=torch.long)
    
    # Calculate final accuracy
    correct_predictions = (majority_votes == y_true).sum().item()
    accuracy = correct_predictions / n_samples
    
    return accuracy




In [78]:
jury_dic = create_jury(n_per_method=5)

--- Assembling the Jury ---

--- Generating 5 vectors from Random Search + Hill Climbing ---

Jury Member 1/5 (Random+Hill)
--- Starting Random Search ---
Iteration 1/10000: New best accuracy = 0.519721
Iteration 2/10000: New best accuracy = 0.552300
Iteration 5/10000: New best accuracy = 0.603267
Iteration 14/10000: New best accuracy = 0.661821
Iteration 5/10000: New best accuracy = 0.603267
Iteration 14/10000: New best accuracy = 0.661821
Iteration 89/10000: New best accuracy = 0.666590
Iteration 89/10000: New best accuracy = 0.666590
Iteration 231/10000: New best accuracy = 0.671599
Iteration 231/10000: New best accuracy = 0.671599
Iteration 347/10000: New best accuracy = 0.673216
Iteration 347/10000: New best accuracy = 0.673216
Iteration 555/10000: New best accuracy = 0.677037
Iteration 555/10000: New best accuracy = 0.677037
Iteration 1283/10000: New best accuracy = 0.684340
Iteration 1283/10000: New best accuracy = 0.684340
Iteration 4857/10000: New best accuracy = 0.695862
Iter

In [89]:
# Calculate average accuracy for each method in the jury
print("=== Average Accuracy by Method ===")

method_accuracies = {}
for method, members in jury_dic.items():
    accuracies = []
    for member_weights in members:
        accuracy = return_accuracy(member_weights)
        accuracies.append(accuracy)
    
    avg_accuracy = np.mean(accuracies)
    method_accuracies[method] = {
        'average': avg_accuracy,
        'individual': accuracies,
        'std': np.std(accuracies),
        'min': np.min(accuracies),
        'max': np.max(accuracies)
    }
    
    print(f"\n{method.upper()}:")
    print(f"  Average accuracy: {avg_accuracy:.6f}")
    print(f"  Standard deviation: {np.std(accuracies):.6f}")
    print(f"  Range: {np.min(accuracies):.6f} - {np.max(accuracies):.6f}")
    print(f"  Individual accuracies: {[f'{acc:.6f}' for acc in accuracies]}")

# Find the best performing method
best_method = max(method_accuracies.keys(), key=lambda x: method_accuracies[x]['average'])
print(f"\n=== SUMMARY ===")
print(f"Best performing method: {best_method.upper()} (avg: {method_accuracies[best_method]['average']:.6f})")

=== Average Accuracy by Method ===

RANDOM_HILL:
  Average accuracy: 0.786273
  Standard deviation: 0.000462
  Range: 0.785789 - 0.787042
  Individual accuracies: ['0.787042', '0.785789', '0.785870', '0.786519', '0.786144']

GENETIC:
  Average accuracy: 0.787060
  Standard deviation: 0.000203
  Range: 0.786800 - 0.787356
  Individual accuracies: ['0.787229', '0.787356', '0.786948', '0.786800', '0.786965']

BAYESIAN:
  Average accuracy: 0.769443
  Standard deviation: 0.007193
  Range: 0.756168 - 0.776837
  Individual accuracies: ['0.770685', '0.774549', '0.756168', '0.776837', '0.768975']

RANDOM_HILL:
  Average accuracy: 0.786273
  Standard deviation: 0.000462
  Range: 0.785789 - 0.787042
  Individual accuracies: ['0.787042', '0.785789', '0.785870', '0.786519', '0.786144']

GENETIC:
  Average accuracy: 0.787060
  Standard deviation: 0.000203
  Range: 0.786800 - 0.787356
  Individual accuracies: ['0.787229', '0.787356', '0.786948', '0.786800', '0.786965']

BAYESIAN:
  Average accuracy: 

In [85]:

for method, members in jury_dic.items():
    jury_members.extend(members)
    print(f"Added {len(members)} members from {method}")

print(f"Total jury members: {len(jury_members)}")

Added 5 members from random_hill
Added 5 members from genetic
Added 5 members from bayesian
Added 5 members from cma_es
Total jury members: 20


In [88]:
# 1. Calculate individual accuracies for each jury member
print("=== Individual Jury Member Accuracies ===")
individual_accuracies = []

for i, member_weights in enumerate(jury_members):
    accuracy = return_accuracy(member_weights)
    individual_accuracies.append(accuracy)
    print(f"Jury Member {i+1}: {accuracy:.6f}")

print(f"\nAverage individual accuracy: {np.mean(individual_accuracies):.6f}")
print(f"Best individual accuracy: {np.max(individual_accuracies):.6f}")
print(f"Worst individual accuracy: {np.min(individual_accuracies):.6f}")
print(f"Standard deviation: {np.std(individual_accuracies):.6f}")

=== Individual Jury Member Accuracies ===
Jury Member 1: 0.787042
Jury Member 2: 0.785789
Jury Member 3: 0.785870
Jury Member 4: 0.786519
Jury Member 5: 0.786144
Jury Member 6: 0.787229
Jury Member 7: 0.787356
Jury Member 8: 0.786948
Jury Member 9: 0.786800
Jury Member 10: 0.786965
Jury Member 11: 0.770685
Jury Member 12: 0.774549
Jury Member 13: 0.756168
Jury Member 14: 0.776837
Jury Member 15: 0.768975
Jury Member 16: 0.787680
Jury Member 17: 0.787718
Jury Member 18: 0.787660
Jury Member 19: 0.787671
Jury Member 20: 0.787729

Average individual accuracy: 0.782617
Best individual accuracy: 0.787729
Worst individual accuracy: 0.756168
Standard deviation: 0.008432
Jury Member 4: 0.786519
Jury Member 5: 0.786144
Jury Member 6: 0.787229
Jury Member 7: 0.787356
Jury Member 8: 0.786948
Jury Member 9: 0.786800
Jury Member 10: 0.786965
Jury Member 11: 0.770685
Jury Member 12: 0.774549
Jury Member 13: 0.756168
Jury Member 14: 0.776837
Jury Member 15: 0.768975
Jury Member 16: 0.787680
Jury Memb

In [92]:
# 2. Calculate and print the final accuracy
final_accuracy = calculate_jury_accuracy(jury_dic["bayesian"], data)

print(f"\n=== Jury System Performance ===")
print(f"Jury Size: {len(jury_members)} members")
print(f"Final Accuracy (Majority Vote): {final_accuracy:.6f}")


=== Jury System Performance ===
Jury Size: 20 members
Final Accuracy (Majority Vote): 0.775612

=== Jury System Performance ===
Jury Size: 20 members
Final Accuracy (Majority Vote): 0.775612


In [84]:
jury_members = []